# 2. Modelos y extensión

**Responsable: Izan.**

La regresión logística inicial procede del cuaderno original de Mónica.
Están pendientes el ajuste, las otras representaciones, la red propia y
la extensión. Primero hay que generar TF-IDF y Word2Vec con el cuaderno 1.
Para comparar BERT se necesitan los vectores del cuaderno 3.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from funciones import (
    RESULTADOS, FIGURAS, cargar_representacion, evaluar, guardar_metricas,
)

FIGURAS.mkdir(parents=True, exist_ok=True)


In [ ]:
from funciones import cargar_datos, preparar_datos, cargar_o_crear_particiones

projects, scivoc = cargar_datos()
df = preparar_datos(projects, scivoc)
conjuntos = cargar_o_crear_particiones(df)
df_train = conjuntos["train"]
df_val = conjuntos["val"]
df_test = conjuntos["test"]


## Primera regresión logística con TF-IDF

Este es el experimento inicial existente, sin ajuste de hiperparámetros.
Se evalúa en validación; sus resultados no se presentan como definitivos.


In [ ]:
matrices_tfidf = cargar_representacion("tfidf", conjuntos)
X_train_tfidf = matrices_tfidf["train"]
X_val_tfidf = matrices_tfidf["val"]
X_test_tfidf = matrices_tfidf["test"]


In [ ]:
modelo_tfidf = LogisticRegression(max_iter=1000, random_state=42)

# Entrenar con los datos de entrenamiento
modelo_tfidf.fit(X_train_tfidf, df_train['label_nivel2'])

In [ ]:
y_val_pred_tfidf = modelo_tfidf.predict(X_val_tfidf)
clases = sorted(df_train["label_nivel2"].unique())
metricas_tfidf = evaluar(df_val["label_nivel2"], y_val_pred_tfidf, clases=clases)
display(pd.DataFrame([metricas_tfidf]))
print(classification_report(
    df_val["label_nivel2"], y_val_pred_tfidf,
    labels=clases, zero_division=0,
))
guardar_metricas(
    "lr_tfidf_inicial", "tfidf", "regresion_logistica",
    2, "val", metricas_tfidf,
)
predicciones = pd.DataFrame({
    "projectID": df_val["projectID"],
    "y_real": df_val["label_nivel2"],
    "y_predicha": y_val_pred_tfidf,
})
(RESULTADOS / "predicciones").mkdir(parents=True, exist_ok=True)
predicciones.to_csv(
    RESULTADOS / "predicciones" / "lr_tfidf_inicial_val_nivel2.csv", index=False
)


In [ ]:
# Incluimos todas las categorías para no ocultar errores fuera del top 10.
fig, ax = plt.subplots(figsize=(13, 11))
ConfusionMatrixDisplay.from_predictions(
    df_val["label_nivel2"], y_val_pred_tfidf, labels=clases,
    normalize="true", cmap="Blues", include_values=False,
    xticks_rotation=90, ax=ax,
)
ax.set_title("Regresión logística inicial: validación, nivel 2")
plt.tight_layout()
plt.savefig(FIGURAS / "confusion_lr_tfidf_inicial_val.png", dpi=150)
plt.show()


## Pendiente: ajustar los modelos clásicos

Empezar con pocas opciones de regularización y, si tiene sentido, ponderación
de clases. Elegir por macro-F1 en validación y registrar lo probado.
El test no se usa para elegir parámetros.

Si se cambian parámetros del vectorizador, volver a ajustar la representación
con train. En validación cruzada, las transformaciones se ajustan dentro
de cada partición, no sobre todo el entrenamiento antes de dividirlo.


In [ ]:
# PENDIENTE: probar configuraciones y guardar la elegida y sus parámetros.


## Pendiente: Word2Vec y BERT congelado

Cargar con cargar_representacion("word2vec", conjuntos) y, cuando Sara lo haya
generado, cargar_representacion("bert", conjuntos).

Comparar las tres representaciones con regresión logística. Para Word2Vec,
ajustar StandardScaler únicamente con train y transformar val y test.
Guardar las métricas con nombres de experimento distintos.


In [ ]:
# PENDIENTE: clasificadores con Word2Vec y BERT congelado.


## Pendiente: red propia con PyTorch

Empezar por una red pequeña sobre vectores densos. Documentar capas,
activaciones, función de pérdida, optimizador, épocas y criterio de parada.
Elegir los parámetros con validación y controlar las semillas.
Si se reduce la dimensión de TF-IDF, aprender la reducción solo con train.


In [ ]:
# PENDIENTE: implementar y entrenar la red propia.


## Pendiente: categorías de nivel 1

Repetir las configuraciones elegidas para las tres representaciones usando
label_nivel1, con los mismos proyectos y particiones. Pasar los resultados
a Sara para comparar el ranking entre niveles.


In [ ]:
# PENDIENTE: entrenar con las etiquetas de nivel 1.


## Pendiente: extensión de texto y metadatos

Comparar texto, metadatos y ambos juntos con los mismos proyectos.
Posibles variables: coste, financiación UE, duración, país coordinador y
número de países participantes. No usar EuroSciVoc ni sus etiquetas como entrada.
La imputación, codificación de países y escalado se ajustan con train.
Elegir la configuración de texto usando validación.


In [ ]:
# PENDIENTE: preparar variables y comparar los tres modelos de la extensión.


## Para cerrar esta parte

Guardar parámetros, métricas y predicciones, escribir la metodología y la
discusión en el README y preparar las diapositivas de esta parte.
Las predicciones finales deben conservar projectID para poder revisar errores.
